# 06 - Evaluation and report

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/demand-forecasting-capstone/blob/main/notebooks/06_evaluation_and_report.ipynb)

**Goal:** compare the agentic system with classical methods fairly, and produce the figures and tables for the report.

Run notebook 05 first for the same dataset.

How to run: `Runtime -> Run all` (Colab) or run the cells from top to bottom. The default dataset is `synthetic`, which needs no downloads.

In [ ]:
# ---- Setup: works in Google Colab and on your own computer. Nothing to upload. ----
REPO_URL = "https://github.com/YOUR-USERNAME/demand-forecasting-capstone.git"   # <- set once, the same in every notebook
BRANCH = "main"
USE_DRIVE = False   # True = use a copy of the project folder in your Google Drive instead of GitHub

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if not os.path.exists("src/demand_agents"):
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.chdir("/content/drive/MyDrive/demand-forecasting-capstone")
    elif IN_COLAB:
        if not os.path.exists("/content/project"):
            subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/project"], check=True)
        os.chdir("/content/project")
    elif os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                                   # running locally from the notebooks folder
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
sys.path.insert(0, os.path.abspath("src"))
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
from IPython.display import display, Image, Markdown
print("Project folder:", os.getcwd())

In [ ]:
DATASET = "synthetic"    # "synthetic" | "synthetic_calm" | "m5" | "m5_christmas" | "walmart" | "walmart_synthetic"
from demand_agents.config import load_named
cfg = load_named(DATASET)
print("dataset:", DATASET, "| source:", cfg["data"]["source"], "| frequency:", cfg["data"]["frequency"])
RUN_SEED_SUITE = False   # 5 independent synthetic worlds (~3 min); only meaningful for synthetic data

## 1. Forecast accuracy and interval quality

**WAPE** = total absolute error / total demand (lower is better). `shock_days` are days the Critic flagged (or known shocks in synthetic data).

In [ ]:
import json
out = cfg["output_dir"]
m = json.load(open(f"{out}/metrics.json"))
display(pd.DataFrame(m["forecast_metrics"]).pivot(index="model", columns="segment", values="wape").round(3))
display(pd.DataFrame(m["interval_metrics"]).round(3))
print("Critic:", {k: round(v, 3) for k, v in m["critic"].items() if isinstance(v, float)})

## 2. Inventory policies (one service-level setting)

Careful: a policy that **holds more stock** looks better here. Use the next section for claims.

In [ ]:
display(pd.read_csv(f"{out}/inventory_summary.csv")[["policy", "fill_rate", "stockout_rate", "avg_on_hand_units", "total_cost"]].round(3))

## 3. The fair comparison: stock needed for the *same* fill rate

**Fill rate** = share of customer demand served. The table shows the average stock each policy needs to reach it. Less stock for the same service is better. `NaN` = never reached.

In [ ]:
from demand_agents.experiments import load_state_from_outputs, run_sensitivity, stock_vs_reference
st = load_state_from_outputs(out)
exp_dir = f"outputs/experiments_{DATASET}"
res = run_sensitivity(st, exp_dir)           # re-plans only; forecasts are not recomputed
display(res["frontier"].pivot(index="target_fill_rate", columns="policy", values="avg_stock_needed").round(1))
ratio = stock_vs_reference(res["service_level"]).set_index("policy")["mean_ratio"].round(3)
display(ratio.to_frame("stock vs gbm_static (1.00 = same, lower = better)"))

## 4. Sensitivity to the assumptions

Costs and lead time are assumptions (see `docs/assumptions.md`). A conclusion that survives these changes is trustworthy.

In [ ]:
for name in ("service_level", "lead_time", "cost_params"):
    x = res[name]; keys = [c for c in x.columns if c in ("service_level", "lead_time", "holding_rate_annual", "stockout_cost_frac")]
    display(Markdown(f"**{name}**: lowest total cost per setting"))
    display(x.loc[x.groupby(keys)["total_cost"].idxmin()][keys + ["policy", "total_cost"]].round(1))

## 5. Figures and a draft report for `reports/`

In [ ]:
os.makedirs("reports/figures", exist_ok=True)
subprocess.run([sys.executable, "scripts/make_report_figures.py", "--results", out, "--experiments", exp_dir, "--out", f"reports/figures/{DATASET}"], check=True)
subprocess.run([sys.executable, "scripts/build_report.py", "--results", out, "--experiments", exp_dir, "--out", f"reports/report_draft_{DATASET}.md"], check=True)
display(Image(f"reports/figures/{DATASET}/fig4_policy_ladder.png"))

## 6. Optional: many synthetic worlds

Repeats everything on independent random worlds, so a result cannot depend on one lucky seed.

In [ ]:
if RUN_SEED_SUITE and cfg["data"]["source"] == "synthetic":
    from demand_agents.experiments import run_seed_suite
    agg = run_seed_suite(cfg, range(101, 106), "outputs/experiments/seeds")
    display(agg[agg["kind"] == "inventory"].round(3))

## 7. Qualitative review sheet

Two teammates rate the agents' explanations (clarity, correctness, usefulness) using `docs/trace_review.md`.

In [ ]:
subprocess.run([sys.executable, "scripts/export_trace_review.py", "--results", out, "--n", "6", "--out", f"reports/trace_review_{DATASET}.csv"], check=True)

## What you can claim

- Compare policies **at equal fill rate**, never at one setting.
- Report real-data and synthetic results separately.
- Say clearly what did **not** help. A true small result beats a doubtful big one.
- Choose settings on validation data and look at the test window once.